# Prep ROG — chapter 1

Convert ROG-Art EXB files into the canonical JSONL.

**Input**
- `data/unpacked/ROG/ROG/...` — annotation files from `ROG.zip`
  - `.exb` files (EXMARaLDA Partitur Editor format, the source we parse here)
  - Per the ROG 1.1 spec, these contain: speaker tiers (`colloq`, `norm`), dialogue act tiers (`dialogueActsIsoDimension`, `dialogueActsIsoFunction`), and sentiment tiers (`sentimentCurated`, `sentimentAnnotated`).
- `data/unpacked/ROG/ROG-Art.wav/...` — the original WAV files (optional for this notebook; the audio splitter uses them in the next step)

**Output**
- `data/processed_jsonl/rog_instance.raw.jsonl` — one canonical-JSONL line per `colloq` segment, with `audio_path` pointing at the *future* cut-WAV location. The audio splitter is what actually creates those WAVs and writes the final `rog_instance.jsonl`.

**What this notebook does NOT do**
- Cut audio. That's `audio_splitter.py`'s job.
- Choose a target label (sentiment vs. dialogue act vs. ...). All labels are written into the `labels` dict; the trainer picks one via its config.

---

## 0. Setup


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "1_data_prep":
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp
PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


PROJECT_ROOT = /home/ivan/Posao_IJS/Stepping Stones/github_full_repo/slavic-speech-pipeline


---

## 1. Config

All knobs are here. Run with `test_mode=True` first to process a handful of files end-to-end.


In [2]:
from dataclasses import dataclass

@dataclass
class Config:
    # Where to find unpacked EXB files. Glob is applied recursively.
    exb_glob: str = "data/unpacked/ROG/ROG/**/*.exb"

    # Where the cut WAVs will eventually live. The audio splitter writes here;
    # we just predict the path so audio_path is correctly populated.
    cut_audio_subdir: str = "data/cut_audio/ROG"

    # Output JSONL (the "raw" suffix indicates audio_path is predictive, not yet realised).
    output_jsonl: str = "data/processed_jsonl/rog_instance.raw.jsonl"

    # Split ratios (train, dev, test). Grouped by file_id so no recording leaks across splits.
    split_ratios: tuple = (0.8, 0.1, 0.1)

    # Drop segments whose `norm` text is missing or shorter than this many chars.
    min_text_chars: int = 2

    # Tolerance for matching tier timestamps. The EXB extractor rounds to 3 decimals,
    # so 0.001 is the natural tolerance for exact-match lookups.
    timestamp_tolerance: float = 0.001

    # Test mode
    test_mode: bool = True
    test_n_files: int = 3   # cap on number of EXB files in test mode

cfg = Config()
print(cfg)

# Tier categories we ask for if present. ROG 1.1 only carries these on the
# Dialog subcorpus; Artur-J / Artur-N files have only "colloq" and will yield
# empty lists for everything else.
WANTED_CATEGORIES = [
    "colloq", "norm",
    "dialogueActsIsoDimension", "dialogueActsIsoFunction",
    "sentimentCurated", "sentimentAnnotated",
]


Config(exb_glob='data/unpacked/ROG/ROG/**/*.exb', cut_audio_subdir='data/cut_audio/ROG', output_jsonl='data/processed_jsonl/rog_instance.raw.jsonl', split_ratios=(0.8, 0.1, 0.1), min_text_chars=2, timestamp_tolerance=0.001, test_mode=True, test_n_files=3)


---

## 2. Find input EXB files

We glob the unpacked ROG folder. If nothing turns up, you probably haven't run `download_data.ipynb` yet, or the unpacked layout is different from what we expect — check the path the glob is hitting.


In [3]:
exb_files = sorted((PROJECT_ROOT).glob(cfg.exb_glob))
print(f"Found {len(exb_files)} EXB files matching {cfg.exb_glob}")
if cfg.test_mode:
    exb_files = exb_files[: cfg.test_n_files]
    print(f"🧪 TEST MODE: capped to {len(exb_files)} files")

for p in exb_files[:5]:
    print(f"  {p.relative_to(PROJECT_ROOT)}")
if len(exb_files) > 5:
    print(f"  ... and {len(exb_files) - 5} more")

if not exb_files:
    raise FileNotFoundError(
        f"No EXB files found at {PROJECT_ROOT / cfg.exb_glob}. "
        "Run download_data.ipynb with dataset='ROG' first, then re-check the glob path."
    )


Found 57 EXB files matching data/unpacked/ROG/ROG/**/*.exb
🧪 TEST MODE: capped to 3 files
  data/unpacked/ROG/ROG/ROG/ROG-Art/EXB/Rog-Art-J-Gvecg-P500001.exb
  data/unpacked/ROG/ROG/ROG/ROG-Art/EXB/Rog-Art-J-Gvecg-P500002.exb
  data/unpacked/ROG/ROG/ROG/ROG-Art/EXB/Rog-Art-J-Gvecg-P500014.exb


---

## 3. Preflight: which tiers exist across all files?

ROG 1.1 ships several subcorpora (Artur-J, Artur-N, Dialog, ParlaMint, ...) and only some carry sentiment and dialogue-act annotations. Before parsing, we scan every EXB and report which categories appear where.

This is the cell to look at if you get "0 intervals" for a label later. If `sentimentCurated` only appears in, say, 200 of 500 files, you know exactly which subcorpus has the annotations and which don't.

In [4]:
from lxml import etree
from collections import Counter, defaultdict

def _quick_subcorpus(doc):
    """Return SUBCORPUS metadata if present, e.g. 'Artur-J'."""
    for ud in doc.findall(".//meta-information//ud-information"):
        if ud.attrib.get("attribute-name") == "SUBCORPUS":
            return ud.text
    return "unknown"


category_to_subcorpora = defaultdict(Counter)  # {category: Counter(subcorpus: n_files_having_it)}
subcorpus_counts = Counter()
files_with_category = Counter()                # {category: n_files_having_it}

for exb_path in exb_files:
    try:
        doc = etree.parse(str(exb_path))
    except Exception as e:
        print(f"⚠️  could not parse {exb_path.name}: {e}")
        continue
    sub = _quick_subcorpus(doc)
    subcorpus_counts[sub] += 1
    cats_here = {t.attrib.get("category") for t in doc.findall(".//tier")}
    for cat in cats_here:
        if cat is None:
            continue
        files_with_category[cat] += 1
        category_to_subcorpora[cat][sub] += 1

print(f"Scanned {len(exb_files)} files\n")
print(f"Subcorpora seen:")
for sub, n in subcorpus_counts.most_common():
    print(f"  {sub:30s} {n} files")

print(f"\nCategory presence across files:")
print(f"  {'category':35s}  files   in_subcorpora")
for cat, n in files_with_category.most_common():
    subs = ", ".join(f"{s}={k}" for s, k in category_to_subcorpora[cat].most_common())
    flag = "  ←  wanted" if cat in WANTED_CATEGORIES else ""
    print(f"  {cat:35s}  {n:5d}   {subs}{flag}")

print(f"\nWanted categories that are MISSING from every file:")
missing = [c for c in WANTED_CATEGORIES if files_with_category[c] == 0]
if not missing:
    print("  (none — all wanted categories appear somewhere)")
else:
    for c in missing:
        print(f"  ⚠️  {c}")

Scanned 3 files

Subcorpora seen:
  Artur-J                        3 files

Category presence across files:
  category                             files   in_subcorpora
  verbalDisfluency                         3   Artur-J=3
  traceability                             3   Artur-J=3
  dialogActsPrimary                        3   Artur-J=3
  disfluencyStructure                      3   Artur-J=3
  notes                                    3   Artur-J=3
  feats                                    3   Artur-J=3
  vocalDisfluency                          3   Artur-J=3
  v                                        3   Artur-J=3
  sentenceId                               3   Artur-J=3
  lemma                                    3   Artur-J=3
  head                                     3   Artur-J=3
  upos                                     3   Artur-J=3
  deprel                                   3   Artur-J=3
  prosodicUnits                            3   Artur-J=3
  colloq                         

---

## 4. EXB parsing primitives

These are ROG-specific helpers. They live in this notebook (not `utils_dataprep.py`) because no other dataset is in EXB format. If a second EXB-based dataset shows up, we lift them out.

The structure of an EXB file:
- A `<timeline>` block maps timeline IDs (`<tli id="T123" time="4.567">`) to seconds.
- Multiple `<tier>` blocks, each scoped to a speaker and a category (e.g. `category="colloq"`).
- Each tier contains `<event start="T120" end="T125">...</event>` items whose text is the annotation for that span.

We parse the timeline once, then for each tier we want, pull all events and resolve their timestamps. To stitch tiers together (colloq + norm + dialogue acts), we use `(start_t, end_t, speaker)` as the key — these tiers are co-aligned in time.


In [5]:
from lxml import etree
from collections import Counter

def get_timeline(doc):
    """Map TLI id → time in seconds."""
    return {
        tli.attrib["id"]: float(tli.attrib["time"])
        for tli in doc.findall(".//tli")
        if "time" in tli.attrib
    }


def list_tier_categories(doc):
    """Return Counter of {category: n_tiers} for every <tier> in the doc.

    Use this to discover what's actually inside an EXB before assuming
    category names. Different ROG subcorpora (Artur-J, Artur-N, Dialog,
    ParlaMint, etc.) carry different annotation layers.
    """
    return Counter(t.attrib.get("category", "<none>") for t in doc.findall(".//tier"))


def extract_tier_intervals(doc, timeline, category):
    """Pull all events for tiers with the given category. Returns list of dicts."""
    intervals = []
    for tier in doc.findall(f'.//tier[@category="{category}"]'):
        speaker = tier.attrib.get("speaker")
        for event in tier.findall("event"):
            sid = event.attrib.get("start")
            eid = event.attrib.get("end")
            st = timeline.get(sid)
            et = timeline.get(eid)
            intervals.append({
                "start_t": round(st, 3) if st is not None else None,
                "end_t":   round(et, 3) if et is not None else None,
                "text":    (event.text or "").strip(),
                "speaker": speaker,
            })
    return intervals


def extract_speaker_metadata(doc):
    """Pull speaker block info (gender, age, etc.)."""
    out = {}
    for sp in doc.findall(".//speaker"):
        sid = sp.attrib.get("id")
        entry = {
            "sex": sp.find("sex").attrib.get("value") if sp.find("sex") is not None else None,
            "abbreviation": sp.findtext("abbreviation"),
        }
        for ud in sp.findall(".//ud-information"):
            name = ud.attrib.get("attribute-name")
            if name:
                entry[name] = ud.text
        out[sid] = entry
    return out


def parse_exb(exb_path: Path):
    """Parse one EXB. Returns (file_id, speakers, tiers_by_category, all_categories)."""
    doc = etree.parse(str(exb_path))
    file_id = exb_path.stem
    timeline = get_timeline(doc)
    speakers = extract_speaker_metadata(doc)
    all_categories = list_tier_categories(doc)
    tiers = {
        cat: extract_tier_intervals(doc, timeline, cat)
        for cat in WANTED_CATEGORIES
    }
    return file_id, speakers, tiers, all_categories


# Smoke test on the first file
if exb_files:
    file_id, speakers, tiers, all_cats = parse_exb(exb_files[0])
    print(f"file_id  = {file_id}")
    print(f"speakers = {list(speakers.keys())}")
    print(f"\nAll tier categories present in this EXB:")
    for cat, n in all_cats.most_common():
        marker = " ✓ (wanted)" if cat in WANTED_CATEGORIES else ""
        print(f"  {cat:35s} {n:3d} tiers{marker}")
    print(f"\nWanted categories — interval counts:")
    for cat, intervals in tiers.items():
        print(f"  {cat:35s} {len(intervals):5d} intervals")

file_id  = Rog-Art-J-Gvecg-P500001
speakers = ['Artur-J-G3003']

All tier categories present in this EXB:
  colloq                                1 tiers ✓ (wanted)
  v                                     1 tiers
  notes                                 1 tiers
  vocalDisfluency                       1 tiers
  verbalDisfluency                      1 tiers
  disfluencyStructure                   1 tiers
  traceability                          1 tiers
  sentenceId                            1 tiers
  lemma                                 1 tiers
  upos                                  1 tiers
  xpos                                  1 tiers
  feats                                 1 tiers
  head                                  1 tiers
  deprel                                1 tiers
  prosodicUnits                         1 tiers
  dialogActsPrimary                     1 tiers

Wanted categories — interval counts:
  colloq                                967 intervals
  norm                 

---

## 5. Stitching tiers into per-segment records

ROG's six tiers split into two groups:

1. **Aligned tiers** (`colloq`, `norm`, `dialogueActsIsoDimension`, `dialogueActsIsoFunction`) — these share exact time spans for the same speaker. We join them on `(start_t, end_t, speaker)`.
2. **Overlap tiers** (`sentimentCurated`, `sentimentAnnotated`) — these have their own boundaries that *contain* one or more `colloq` segments. We find which sentiment interval each `colloq` segment falls inside.

The result is one record per `colloq` segment, with every available label attached.


In [6]:
def _make_key(interval):
    s, e, sp = interval.get("start_t"), interval.get("end_t"), interval.get("speaker")
    if s is None or e is None:
        return None
    return (round(s, 3), round(e, 3), sp)


def _find_containing_sentiment(seg, sentiment_intervals, tol):
    """Return the sentiment label whose span contains seg, or None."""
    s, e = seg.get("start_t"), seg.get("end_t")
    if s is None or e is None:
        return None
    for sent in sentiment_intervals:
        ss, ee = sent.get("start_t"), sent.get("end_t")
        if ss is None or ee is None:
            continue
        if ss - tol <= s <= ee + tol and ss - tol <= e <= ee + tol:
            return sent.get("text") or None
    return None


def stitch_segments(file_id, speakers, tiers, tol=0.001):
    """Build one record per colloq segment, with all matching labels."""
    norm_lookup = {k: v for v in tiers["norm"] if (k := _make_key(v))}
    dim_lookup  = {k: v for v in tiers["dialogueActsIsoDimension"] if (k := _make_key(v))}
    func_lookup = {k: v for v in tiers["dialogueActsIsoFunction"] if (k := _make_key(v))}

    out = []
    for colloq in tiers["colloq"]:
        key = _make_key(colloq)
        if key is None:
            continue

        norm = norm_lookup.get(key)
        dim  = dim_lookup.get(key)
        func = func_lookup.get(key)

        out.append({
            "file_id": file_id,
            "speaker": colloq["speaker"],
            "start_t": colloq["start_t"],
            "end_t":   colloq["end_t"],
            "colloq_text": colloq["text"],
            "norm_text":   norm["text"] if norm else None,
            "dialogue_act_dimension": (dim["text"] if dim else None) or None,
            "dialogue_act_function":  (func["text"] if func else None) or None,
            "sentiment_curated":   _find_containing_sentiment(colloq, tiers["sentimentCurated"], tol),
            "sentiment_annotated": _find_containing_sentiment(colloq, tiers["sentimentAnnotated"], tol),
            "speaker_metadata": speakers.get(colloq["speaker"], {}),
        })
    return out


# Smoke test
if exb_files:
    file_id, speakers, tiers, _all_cats = parse_exb(exb_files[0])
    segs = stitch_segments(file_id, speakers, tiers, tol=cfg.timestamp_tolerance)
    print(f"Stitched {len(segs)} segments from {file_id}")
    if segs:
        import json as _json
        print("First segment:")
        print(_json.dumps(segs[0], ensure_ascii=False, indent=2))


Stitched 967 segments from Rog-Art-J-Gvecg-P500001
First segment:
{
  "file_id": "Rog-Art-J-Gvecg-P500001",
  "speaker": "Artur-J-G3003",
  "start_t": 2082.198,
  "end_t": 2082.453,
  "colloq_text": "Drage",
  "norm_text": null,
  "dialogue_act_dimension": null,
  "dialogue_act_function": null,
  "sentiment_curated": null,
  "sentiment_annotated": null,
  "speaker_metadata": {
    "sex": "m",
    "abbreviation": "Artur-J-G3003",
    "PRS-ID": "Artur-J-G3003",
    "SEX": "moški",
    "AGE": "30 do 59 let",
    "1LANG": "slovenščina",
    "DIALECT": "standardni jezik",
    "EDUCATION": "fakulteta ali več",
    "PERM-RESD": "osrednjeslovenska",
    "CHILD-RESD": "-",
    "SOURCE-ID": "Artur-J-Gvecg-P500001",
    "RECORDING-ID": "Artur-J-Gvecg-P500001.wav"
  }
}


---

## 6. Mapping to canonical JSONL records

Now we take the stitched segments and emit canonical-JSONL dicts:
- `instance_id` via `udp.make_instance_id`
- `audio_path` predicted (the splitter will create this file later)
- `labels` dict carries everything we extracted
- `metadata` dict carries speaker info and source provenance


In [7]:
def to_canonical(segment: dict, cfg: Config) -> dict:
    speaker = segment["speaker"]
    file_id = segment["file_id"]
    st, et = segment["start_t"], segment["end_t"]

    iid = udp.make_instance_id("ROG", file_id, speaker, st, et)

    # Predict the future cut-WAV path so the splitter knows where to write.
    cut_name = f"{file_id}_{speaker}_{st:.3f}_{et:.3f}.wav"
    audio_path = f"{cfg.cut_audio_subdir}/{cut_name}"

    # Use norm if present, else colloq. Records with neither are dropped later by clean().
    text = segment.get("norm_text") or segment.get("colloq_text") or ""

    labels = {}
    if segment.get("sentiment_curated"):
        labels["sentiment"] = segment["sentiment_curated"]
    elif segment.get("sentiment_annotated"):
        # Fall back to the annotator value if curated is missing
        labels["sentiment"] = segment["sentiment_annotated"]
    if segment.get("sentiment_annotated"):
        labels["sentiment_annotated"] = segment["sentiment_annotated"]
    if segment.get("dialogue_act_function"):
        labels["dialogue_act_function"] = segment["dialogue_act_function"]
    if segment.get("dialogue_act_dimension"):
        labels["dialogue_act_dimension"] = segment["dialogue_act_dimension"]

    metadata = {
        "source_file": f"{file_id}.exb",
        "source_format": "EXB",
        "speaker_info": segment.get("speaker_metadata", {}),
        "colloq_text": segment.get("colloq_text"),  # keep original colloquial form for reference
    }

    return {
        "instance_id": iid,
        "dataset": "ROG",
        "file_id": file_id,
        "audio_path": audio_path,
        # split is set later by udp.assign_splits — placeholder for now
        "split": "train",
        "speaker": speaker,
        "start_t": st,
        "end_t": et,
        "text": text,
        "labels": labels,
        "metadata": metadata,
    }


# Smoke test
if exb_files:
    sample = to_canonical(segs[0], cfg)
    errs = udp.validate_instance(sample)
    if errs:
        print("⚠️  validation errors on sample:")
        for e in errs:
            print(f"   - {e}")
    else:
        print("✅ sample validates against canonical schema")
    import json as _json
    print(_json.dumps(sample, ensure_ascii=False, indent=2))


✅ sample validates against canonical schema
{
  "instance_id": "ROG_Rog-Art-J-Gvecg-P500001_Artur-J-G3003_2082.198_2082.453",
  "dataset": "ROG",
  "file_id": "Rog-Art-J-Gvecg-P500001",
  "audio_path": "data/cut_audio/ROG/Rog-Art-J-Gvecg-P500001_Artur-J-G3003_2082.198_2082.453.wav",
  "split": "train",
  "speaker": "Artur-J-G3003",
  "start_t": 2082.198,
  "end_t": 2082.453,
  "text": "Drage",
  "labels": {},
  "metadata": {
    "source_file": "Rog-Art-J-Gvecg-P500001.exb",
    "source_format": "EXB",
    "speaker_info": {
      "sex": "m",
      "abbreviation": "Artur-J-G3003",
      "PRS-ID": "Artur-J-G3003",
      "SEX": "moški",
      "AGE": "30 do 59 let",
      "1LANG": "slovenščina",
      "DIALECT": "standardni jezik",
      "EDUCATION": "fakulteta ali več",
      "PERM-RESD": "osrednjeslovenska",
      "CHILD-RESD": "-",
      "SOURCE-ID": "Artur-J-Gvecg-P500001",
      "RECORDING-ID": "Artur-J-Gvecg-P500001.wav"
    },
    "colloq_text": "Drage"
  }
}


---

## 7. Run over all files

Process every EXB, stitch, map to canonical, accumulate.


In [8]:
records = []
file_stats = []

for exb_path in exb_files:
    try:
        file_id, speakers, tiers, _all_cats = parse_exb(exb_path)
        segs = stitch_segments(file_id, speakers, tiers, tol=cfg.timestamp_tolerance)
        these = [to_canonical(s, cfg) for s in segs]
        records.extend(these)
        file_stats.append({
            "file_id": file_id,
            "n_segments": len(these),
            "n_with_sentiment": sum(1 for r in these if "sentiment" in r["labels"]),
            "n_with_dialogue_act": sum(1 for r in these if "dialogue_act_function" in r["labels"]),
        })
        print(f"  {file_id:50s} {len(these):4d} segs   sent={file_stats[-1]['n_with_sentiment']:4d}   da={file_stats[-1]['n_with_dialogue_act']:4d}")
    except Exception as e:
        print(f"  ❌ {exb_path.name}: {e}")

print(f"\nTotal records before cleaning: {len(records)}")
print(f"Of those: with sentiment    = {sum(s['n_with_sentiment'] for s in file_stats)}")
print(f"          with dialogue act = {sum(s['n_with_dialogue_act'] for s in file_stats)}")

  Rog-Art-J-Gvecg-P500001                             967 segs   sent=   0   da=   0
  Rog-Art-J-Gvecg-P500002                             819 segs   sent=   0   da=   0
  Rog-Art-J-Gvecg-P500014                             929 segs   sent=   0   da=   0

Total records before cleaning: 2715
Of those: with sentiment    = 0
          with dialogue act = 0


---

## 8. Clean and validate

`udp.clean` runs the standard pipeline: drop invalid records, drop duplicates by `instance_id`, drop entries with empty text. We do NOT drop by missing label here — that's the trainer's job at load time (it knows which `label_key` it wants).


In [9]:
records = udp.clean(records, require_text=True, verbose=True)

# Filter by min text length too
before = len(records)
records = [r for r in records if len(r.get("text", "")) >= cfg.min_text_chars]
print(f"  min_text_chars={cfg.min_text_chars}: -{before - len(records)} ({len(records)} left)")

# Final full validation pass — strict-ish: report errors but keep going so we can see them all
n_total, n_valid, errs = udp.validate_jsonl(records, max_report=10)
print(f"\nValidation: {n_valid}/{n_total} valid")
if errs:
    print("First errors:")
    for e in errs:
        print(f"  - {e}")


  drop_invalid:        -0 (2715 left)
  drop_duplicates:     -6 (2709 left)
  drop_empty_text:     -0 (2709 left)
  clean: 2715 → 2709 (6 dropped total)
  min_text_chars=2: -417 (2292 left)

Validation: 2292/2292 valid


---

## 9. Assign splits

Splits are deterministic (md5 over file_id) and grouped by file_id so no recording leaks across train/dev/test.


In [10]:
udp.assign_splits(records, ratios=cfg.split_ratios, group_key="file_id", overwrite=True)
counts = udp.split_summary(records)
print(f"Split counts: {counts}")

# Sanity check: same file_id must always have the same split
groups = {}
for r in records:
    groups.setdefault(r["file_id"], set()).add(r["split"])
leaks = {f: s for f, s in groups.items() if len(s) > 1}
print(f"Files with leakage across splits: {len(leaks)}")
assert not leaks, f"split leakage detected: {leaks}"
print("✅ no split leakage")


Split counts: {'train': 797, 'dev': 702, 'test': 793, 'other': 0}
Files with leakage across splits: 0
✅ no split leakage


---

## 10. Label distributions (quick peek)

Just so we can see what we're sitting on before training. Full sniffing happens in chapter 2.


In [11]:
from collections import Counter

def show_label(records, key):
    c = Counter(r["labels"].get(key) for r in records)
    print(f"\n{key}:")
    for k, v in c.most_common():
        bar = "█" * min(40, v * 40 // max(1, max(c.values())))
        print(f"  {str(k):35s} {v:5d}  {bar}")

show_label(records, "sentiment")
show_label(records, "dialogue_act_function")
show_label(records, "dialogue_act_dimension")



sentiment:
  None                                 2292  ████████████████████████████████████████

dialogue_act_function:
  None                                 2292  ████████████████████████████████████████

dialogue_act_dimension:
  None                                 2292  ████████████████████████████████████████


---

## 11. Write the canonical JSONL


In [12]:
out_path = PROJECT_ROOT / cfg.output_jsonl
if cfg.test_mode:
    # Don't pollute real outputs in test mode
    out_path = out_path.with_name("test_" + out_path.name)

n_written = udp.write_jsonl(records, out_path)
print(f"✅ wrote {n_written} records to {out_path.relative_to(PROJECT_ROOT)}")

# Quick re-read sanity check
roundtripped = udp.read_jsonl(out_path)
assert len(roundtripped) == n_written
print(f"✅ round-trip read confirms {len(roundtripped)} records")


✅ wrote 2292 records to data/processed_jsonl/test_rog_instance.raw.jsonl
✅ round-trip read confirms 2292 records


---

## Next

Run `audio_splitter.py` on this JSONL to cut per-instance WAVs and produce the final `rog_instance.jsonl` (without the `.raw` suffix).

```bash
python audio_splitter.py \
    --jsonl data/processed_jsonl/rog_instance.raw.jsonl \
    --out   data/processed_jsonl/rog_instance.jsonl
```

Then point `2_data_analysis/sniff_dataset.py` at it to look at the distributions properly.

If anything in this notebook surprised you (e.g. very low sentiment coverage), open `data/unpacked/ROG/...` and inspect a raw EXB in EXMARaLDA — the source is the ground truth, not this notebook.
